In [3]:
# Reads watermark JSON and appends new OneLake static CSV rows to Bronze tables.

import json, os
from pyspark.sql.functions import col, upper, trim

MOUNT_BASE = "/lakehouse/default/Files"
WATERMARK_PATH = f"{MOUNT_BASE}/bronze/_watermark/watermark.json"
STATIC_MOUNT_PATH = f"{MOUNT_BASE}/source/onelake_static"
STATIC_SPARK_PATH = "Files/source/onelake_static"

print("Files available in source/onelake_static:")

try:
    for item in notebookutils.fs.ls("Files/source/onelake_static"):
        print(item.name)
except Exception as e:
    print("Could not list using relative Lakehouse path.")
    print(str(e))

DEFAULT_WATERMARK = {
    "dim_terminal": "1900-01-01T00:00:00Z",
    "dim_product": "1900-01-01T00:00:00Z",
    "customer_contracts": "1900-01-01T00:00:00Z",
    "terminal_movements": "1900-01-01T00:00:00Z"
}

os.makedirs(os.path.dirname(WATERMARK_PATH), exist_ok=True)

if os.path.exists(WATERMARK_PATH):
    with open(WATERMARK_PATH, "r") as f:
        watermark = json.load(f)
else:
    watermark = DEFAULT_WATERMARK.copy()
    with open(WATERMARK_PATH, "w") as f:
        json.dump(watermark, f, indent=2)

def append_static_csv(file_name, table_name, key_col, watermark_key):
    spark_read_path = f"{STATIC_SPARK_PATH}/{file_name}"
    local_check_path = f"{STATIC_MOUNT_PATH}/{file_name}"

    if not os.path.exists(local_check_path):
        print(f"Missing file: {local_check_path}")
        return 0

    df = spark.read.option("header", True).csv(spark_read_path)
    df = df.withColumn(key_col, upper(trim(col(key_col))))
    df = df.filter(col("last_modified_utc") > watermark[watermark_key])
    row_count = df.count()

    if row_count > 0:
        df.write.format("delta").mode("append").saveAsTable(table_name)

    print(f"{table_name}: {row_count} new or changed rows")
    return row_count

terminal_rows = append_static_csv("dim_terminal.csv", "bronze_terminal_raw", "terminal_code", "dim_terminal")
product_rows = append_static_csv("dim_product.csv", "bronze_product_raw", "product_code", "dim_product")

result = watermark.copy()
result["static_terminal_rows"] = terminal_rows
result["static_product_rows"] = product_rows
result["run_copy_sources"] = True

notebookutils.notebook.exit(json.dumps(result))

StatementMeta(, 41f3dd1b-2301-479f-8dc5-d63794a61de9, 5, Finished, Available, Finished, False)

Files available in source/onelake_static:
dim_product.csv
dim_terminal.csv
bronze_terminal_raw: 8 new or changed rows
bronze_product_raw: 9 new or changed rows
ExitValue: {"dim_terminal": "1900-01-01T00:00:00Z", "dim_product": "1900-01-01T00:00:00Z", "customer_contracts": "1900-01-01T00:00:00Z", "terminal_movements": "1900-01-01T00:00:00Z", "static_terminal_rows": 8, "static_product_rows": 9, "run_copy_sources": true}